In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import sqlite3
from datetime import datetime, timedelta

# Initialize SQLite database with some tables
def init_database():
    conn = sqlite3.connect('warehouse2.db')
    c = conn.cursor()

    c.execute('''CREATE TABLE IF NOT EXISTS inventory
                (product_id TEXT PRIMARY KEY,
                name TEXT,
                quantity INTEGER,
                reorder_point INTEGER,
                unit_price REAL)''')

    c.execute('''CREATE TABLE IF NOT EXISTS orders
                (order_id TEXT PRIMARY KEY,
                order_date TIMESTAMP,
                status TEXT,
                total_amount REAL)''')

    c.execute('''CREATE TABLE IF NOT EXISTS order_items
                (order_id TEXT,
                product_id TEXT,
                quantity INTEGER,
                FOREIGN KEY(order_id) REFERENCES orders(order_id),
                FOREIGN KEY(product_id) REFERENCES inventory(product_id))''')
    
    c.execute('DELETE FROM inventory')
    c.execute('DELETE FROM orders')
    c.execute('DELETE FROM order_items')

    # Sample inventory
    inventory_data = [
        ('SKU001', 'Gaming Laptop', 45, 20, 999.99),
        ('SKU002', 'Wireless Mouse', 150, 50, 29.99),
        ('SKU003', 'Mechanical Keyboard', 5, 30, 89.99),
        ('SKU004', 'Monitor 27"', 30, 15, 299.99),
        ('SKU005', 'USB-C Hub', 10, 40, 49.99)
    ]
    c.executemany('INSERT INTO inventory VALUES (?,?,?,?,?)', inventory_data)

    for i in range(1, 21):  # Create 20 sample orders
        order_id = f'ORD{i:03d}'
        order_date = datetime.now() - timedelta(days=i)
        status = ['Pending', 'Processing', 'Shipped', 'Delivered'][i % 4]
        c.execute('INSERT INTO orders VALUES (?,?,?,?)',
              (order_id, order_date, status, 0)) # Total will be updated

    # Add 1-3 items to each order
    for j in range(1, (i % 3) + 2):
        product_id = f'SKU{(i + j) % 5 + 1:03d}'
        quantity = (i + j) % 5 + 1
        c.execute('INSERT INTO order_items VALUES (?,?,?)',
                  (order_id, product_id, quantity))
        
    conn.commit()
    conn.close()
    
init_database()

C:\Users\Aditya\AppData\Local\Temp\ipykernel_23208\478722823.py:47: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  c.execute('INSERT INTO orders VALUES (?,?,?,?)',


In [3]:
from langchain_community.utilities.sql_database import SQLDatabase

from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_experimental.tools import PythonREPLTool

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
# Create database connection
db = SQLDatabase.from_uri("sqlite:///warehouse2.db")

# Create the SQL toolkit
sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Create tool instances
python_repl = PythonREPLTool()

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
members = ["InventoryManager", "OrderProcessor", "AnalyticsManager"]

tasks_handled = [
    "InventoryManager can check stock levels, update inventory, and manage reordering",
    "OrderProcessor can process orders, update order status, and manage order fulfillment",
    "AnalyticsManager can analyze sales trends, inventory turnover, and generate reports"
]

member_tasks = {member: task for member, task in zip(members, tasks_handled)}

member_tasks

{'InventoryManager': 'InventoryManager can check stock levels, update inventory, and manage reordering',
 'OrderProcessor': 'OrderProcessor can process orders, update order status, and manage order fulfillment',
 'AnalyticsManager': 'AnalyticsManager can analyze sales trends, inventory turnover, and generate reports'}

In [5]:
"\n".join([f"{member}:{task}" for member,task in member_tasks.items()])

'InventoryManager:InventoryManager can check stock levels, update inventory, and manage reordering\nOrderProcessor:OrderProcessor can process orders, update order status, and manage order fulfillment\nAnalyticsManager:AnalyticsManager can analyze sales trends, inventory turnover, and generate reports'

In [6]:
system_prompt = (
    "You are a warehouse operations supervisor managing these workers: {members}."
    " Below are the tasks handled by each worker:\n" +
    "\n".join([f"{member}: {task}" for member, task in member_tasks.items()]) + "\n"
    " Given the following request, determine which worker should act next."
    " When all necessary tasks are completed, respond with FINISH."
)

system_prompt

options = ["FINISH"]+members

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="messages"),
    ("system", "Given the conversation above, who should act next?"
     " Or should we FINISH? Select one of: {options}"),
]).partial(options=str(options), members=", ".join(members))

prompt

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [ ]:
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.output_parsers import StrOutputParser

def supervisor_agent(state):
    supervisorchain = prompt | llm | StrOutputParser()
    
    return supervisorchain.invoke(state)

supervisor_agent(
    {
        "messages":[
            HumanMessage(content="Check if we have enough inventory to fullfill order ORD005  and process if it is possible")
        ]
    }
)

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent

inventory_agent = create_react_agent(
    llm,
    [python_repl, *sql_toolkit.get_tools()],
    prompt="""You are the Inventory Manager. You can:
1. Query inventory levels using SQL
2. Update inventory quantities
3. Check items below reorder point
4. if generating code, make sure to use the corre , make sure that code returns expected value finally.
think properly before generating code and generate all the required code at once.

Important Instructions:
- Always end your response with a clear CONCLUSION: stating what you found and any actions taken
- When checking inventory, provide specific numbers
- For updates, confirm the before and after quantities
- Keep responses focused and concise
- Use proper SQL syntax and verify data before updates

Example format:
*Analysis/Actions taken*
CONCLUSION: Found X items below reorder point. Updated inventory for item Y from A to B units."""
, debug=False
)

In [ ]:
order_agent = create_react_agent(
    llm,
    [python_repl, *sql_toolkit.get_tools()],
    prompt="""You are the Order Processor. You can:
1. Process new orders
2. Update order status
3. Check order details
4. Verify inventory availability
Important Instructions:
- Always end your response with a clear CONCLUSION: stating what you found and any actions taken
- Verify inventory before processing orders
- For order updates, confirm the new status
- Provide specific quantities and order details
- Keep responses focused and concise
Example format:
*Analysis/Actions taken*
CONCLUSION: Order XYZ processed successfully. Updated status from A to B. Inventory adjusted."""
, debug=False
)

In [ ]:
analytics_agent = create_react_agent(
    llm,
    [python_repl, *sql_toolkit.get_tools()],
    prompt="""You are the Analytics Manager. You can:
1. Analyze sales trends
2. Generate inventory reports
3. Calculate key metrics
4. Provide recommendations

Important Instructions:
- Always end your response with a clear CONCLUSION: stating your findings and recommendations
- Provide specific numbers and insights
- Keep responses focused and concise
- Use clear visualizations when needed

Example format:
*Analysis performed*
CONCLUSION: Analysis shows X trend over Y period. Recommend taking Z action."""
,debug=False
)